In [ ]:
# Imports and setup
import os
import re
import string
from collections import Counter

import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
# NLTK Download  
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\arafa\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

In [47]:
# Configuration
chat_folder = "H:\Project and papers work\AI Chat Log Summarizer"

In [48]:
print(chat_folder)

H:\Project and papers work\AI Chat Log Summarizer


In [56]:
# Load & parse every chat log
all_user_msgs = []
all_ai_msgs   = []
raw_docs      = []

for fname in os.listdir(chat_folder):
    if not fname.lower().endswith(".txt"):
        continue
    path = os.path.join(chat_folder, fname)


    # 3b. Parse line-by-line for stats & keywords
    with open(path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    for line in lines:
        text = line.strip()
        if text.startswith("User:"):
            all_user_msgs.append(text[len("User:"):].strip())
        elif text.startswith("AI:"):
            all_ai_msgs.append(text[len("AI:"):].strip())

In [50]:
# Message stat
total_user = len(all_user_msgs)
total_ai   = len(all_ai_msgs)
total_msgs = total_user + total_ai

print(f"Total messages: {total_msgs}")
print(f" - User messages: {total_user}")
print(f" - AI messages:   {total_ai}")

Total messages: 800
 - User messages: 400
 - AI messages:   400


In [52]:
# keyword extraction
all_text = " ".join(all_user_msgs + all_ai_msgs).lower()
words    = re.findall(r'\b\w+\b', all_text)
stops    = set(stopwords.words('english'))
filtered = [w for w in words if w not in stops and len(w) > 1]
counts   = Counter(filtered)
top5_simple = [w for w, _ in counts.most_common(5)]
print("\nTop most 5 keywords (simple count):", top5_simple)


Top most 5 keywords (simple count): ['topic', 'use', 'hi', 'tell', 'sure']


In [53]:
# TF-IDF keyword extraction
corpus    = all_user_msgs + all_ai_msgs
vectorizer = TfidfVectorizer(stop_words='english')
matrix     = vectorizer.fit_transform(corpus)
features   = vectorizer.get_feature_names_out()
scores     = matrix.sum(axis=0).A1
feat_score = list(zip(features, scores))
feat_score.sort(key=lambda x: x[1], reverse=True)
top5_tfidf = [kw for kw, _ in feat_score[:5]]
print("5 keywords of TF-IDF:   ", top5_tfidf)

5 keywords of TF-IDF:    ['topic', 'use', 'hi', 'tell', 'applications']


In [54]:
# summary
print("\nSUMMARY")
print(f"- The conversation had {total_msgs} exchanges.")
main_topics = top5_tfidf[:2] or top5_simple[:2]
print(f"- The user asked mainly about {', '.join(main_topics)}.")
chosen = top5_tfidf if top5_tfidf else top5_simple
print(f"- Most common keywords: {', '.join(chosen)}")


SUMMARY
- The conversation had 800 exchanges.
- The user asked mainly about topic, use.
- Most common keywords: topic, use, hi, tell, applications
